In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
def get_directories(path):
    directories = []
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_dir():
                directories.append(entry.name)
    return np.sort(directories)


path = './'
folders = get_directories(path)


folders = sorted(folders)
folders =folders[0:3]

# print the sorted array
print(folders)

forces=[0]*len(folders)
energies=[0]*len(folders)

for i, dir in enumerate(folders):
    forces[i]=np.transpose(np.loadtxt('{}/{}/train/dataset/forces.comp'.format(path,dir)))
    energies[i]=np.transpose(np.loadtxt('{}/{}/train/dataset/energy.comp'.format(path,dir)))
    

In [ ]:
f_structures=[]  #Number of the config
f_atom_index=[]  #Label of the atom in the system: they are organized as follows: first are H then O then Fe
f_ref=[]         #DFT force
f_NN=[]          #Predicted NN force
e_structures=[]  
e_ref=[]
e_NN=[]




e_diff=[0]*len(folders)             #difference reference DFT and NN prediction
f_diff=[0]*len(folders)
e_bound=0.005                       #Energy threeshold 
f_bound=2.5                       #Force threeshold
e_outlayer_index=[0]*len(folders)    #Line number of the out of bounds configs
f_outlayer_index=[0]*len(folders)



for i, dir in enumerate(folders):
    f_structures.append(forces[i][0])
    e_structures.append(energies[i][0])
    f_atom_index.append(forces[i][1])
    f_ref.append(forces[i][2])
    e_ref.append(energies[i][2]/energies[i][1])
    f_NN.append(forces[i][3])
    e_NN.append(energies[i][3]/energies[i][1])


for i, dir in enumerate(folders):
    e_diff[i]=e_ref[i]-e_NN[i]
    f_diff[i]=f_ref[i]-f_NN[i]

    e_outlayer_index[i]=np.where(np.abs(e_diff[i]) > e_bound)[0]
    f_outlayer_index[i]=np.where(np.abs(f_diff[i]) > f_bound)[0]


In [ ]:
for i, dir in enumerate(folders):
    print(f'Number of energy and force outlayers in {dir}     ' , len(e_outlayer_index[i]),  len(f_outlayer_index[i]))
    

In [ ]:
for i, dir in enumerate(folders):
    print(f'Fract. of energy and force outlayers in {dir}',  f"{len(e_outlayer_index[i])/len(e_structures[i]):.5f}", f"{len(f_outlayer_index[i])/len(f_structures[i]):.5f}")

In [ ]:
#PLOT ENERGIES

x = np.linspace(-4.5, -4.1)

for i, dir in enumerate(folders):
    plt.figure(i+1)
    plt.scatter(energies[i][2]/energies[i][1],energies[i][3]/energies[i][1], s = 0.1)
    plt.xlabel('Reference')
    plt.ylabel('NN')
    plt.title('Energies {}'.format(dir))
    plt.plot(x, x)
    plt.ylim(-4.40,-4.16)
    plt.xlim(-4.40,-4.16)
    plt.plot(x, x+e_bound)
    plt.plot(x, x-e_bound)
    

In [ ]:
#PLOT FORCES
y = np.linspace(-20 ,20)
for i, dir in enumerate(folders):
    plt.figure(i+1)
    plt.scatter(forces[i][2],forces[i][3], s = 0.5)
    plt.xlabel('Reference')
    plt.ylabel('NN')
    plt.ylim(-20,20)
    plt.xlim(-20,20)
    plt.title('Forces {}'.format(dir))
    plt.plot(y, y+f_bound)
    plt.plot(y, y- f_bound)
    plt.plot(y, y)


In [ ]:
#PLOT ENERGIES AND FORCES


# Loop through the folders and plot the energies and forces on each subplot
for i, dir in enumerate(folders):
    # Create a grid of subplots with one row and two columns
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    # Plot the energies on the left subplot
    axs[0].scatter(energies[i][2]/energies[i][1], energies[i][3]/energies[i][1], s=0.1)
    axs[0].set_xlabel('Reference')
    axs[0].set_ylabel('NN')
    axs[0].set_title('Energies {}'.format(dir))
    axs[0].plot(x, x)
    axs[0].set_ylim(-4.4, -4.33)
    axs[0].set_xlim(-4.4, -4.33)
    axs[0].plot(x, x+e_bound)
    axs[0].plot(x, x-e_bound)
    
    #col = np.zeros(np.array(forces[i][3]).shape[0])
    # col[0] = energy[0]
    # col[1] = energy[0]
    # col[2] = energy[0]
    # col[3] = energy[1]
    
    
    
    # Plot the forces on the right subplot
    axs[1].scatter(forces[i][2], forces[i][3],  s=0.1)
    axs[1].set_xlabel('Reference')
    axs[1].set_ylabel('NN')
    axs[1].set_ylim(-6, 6)
    axs[1].set_xlim(-6, 6)
    axs[1].set_title('Forces {}'.format(dir))
    axs[1].plot(y, y+f_bound)
    axs[1].plot(y, y-f_bound)
    axs[1].plot(y, y)

# Show the plot
plt.show()



In [ ]:
#select the outlayers
e_selected_structures= [0]*len(folders)
f_selected_structures = [0]*len(folders)
f_selected_atom = [0]*len(folders)
e_selected_energies = [0]*len(folders)


for j, dir in enumerate(folders): 
    e_selected_structures[j] = [e_structures[j][i] for i in e_outlayer_index[j]]
    e_selected_energies[j]=[e_ref[j][i] for i in e_outlayer_index[j]]
    f_selected_atom[j] = [f_atom_index[j][i] for i in f_outlayer_index[j]]
    f_selected_structures[j] = [f_structures[j][i] for i in f_outlayer_index[j]]


In [ ]:
#force analysis
f_structure_unique=[0]*len(folders)
counts=[0]*len(folders)
for i, dir in enumerate(folders): 
    # Use 'np.unique' with the 'return_counts' argument to get the number of occurrences of each copy
    f_structure_unique[i], counts[i] = np.unique(f_selected_structures[i], return_counts=True)

# Print the result
for i, dir in enumerate(folders):
    #print()
    
    for struc, count in zip(f_structure_unique[i], counts[i]):
        if count >= 4:
            print(f"In Folder {dir}:")
            print("Structure", int(struc), "occurs", count, "times")

In [ ]:
f_selected_energies_all=[]
f_selected_structure_all=[]
f_selected_energies=[0]*len(folders)
for i, dir in enumerate(folders):
    print(dir)
    print(f_structure_unique[i])
    f_selected_structure_all.append(f_structure_unique[i])
    tmp=[]
    for k in range(len(f_structure_unique[i])):
        #print(f_structure_unique[i][k])
        for line, str in enumerate(energies[i][0]):
            if str == f_structure_unique[i][k]:
                #print(int(line))
                
                #print(f_structure_unique[i][k],energies[i][2][int(line)])
                #f_selected_energies_all.append(energies[i][2][int(line)])
                tmp.append(energies[i][2][int(line)])
    f_selected_energies[i]=tmp
    f_selected_energies_all.append(tmp)
        

In [ ]:
#force analysis: intersection of outlayers in various run 

#intersection of force based structures 
common_structures= set(f_selected_structure_all[0]).intersection(*f_selected_structure_all[1:])
#intersection of force based energies
common_energies=set(f_selected_energies_all[0]).intersection(*f_selected_energies_all[1:])

print(len(common_structures))

list_common_f=[]
for ene in common_energies:
    #print(f_structure_unique[0][f_selected_energies[0].index(ene)], ene)
    list_common_f.append(int(f_structure_unique[0][f_selected_energies[0].index(ene)]))
list_common_f.sort()
 
print(list_common_f)
#untag for save list of bad config for the script "print_config_input_data.ipynb"
#np.savetxt( "bad_list_intersection", list_common_f, fmt='%d' )   


In [ ]:
#union of outlayers in various run force analysis
#union of f structures 
union_structures= set(f_selected_structure_all[0]).union(*f_selected_structure_all[1:])
#union of f structures 
union_energies=set(f_selected_energies_all[0]).union(*f_selected_energies_all[1:])



list_union_f=list(union_structures)
list_union_f.sort()
print(len(list_union_f))
print(list_union_f)
#untag for save list of bad config for the script "print_config_input_data.ipynb"
#np.savetxt( "bad_list_union", list_union_f, fmt='%d' )

In [ ]:
#energy based analysis

#union  over the runs  
union_e_structures= set(e_selected_structures[0]).union(*e_selected_structures[1:])
union_e_energies=set(e_selected_energies[0]).union(*e_selected_energies[1:])

print( "Number union outlayers =", len(union_e_energies))


list_union_e=list(union_e_structures)
list_union_e.sort()
#print(list_union_e)


#intersection over the runs
intersection_e_structures= set(e_selected_structures[0]).intersection(*e_selected_structures[1:])
intersection_e_energies=set(e_selected_energies[0]).intersection(*e_selected_energies[1:])

print("Number intersection outlayers =" , len(intersection_e_energies))
list_intersection_e=list(intersection_e_structures)
list_intersection_e.sort()

In [ ]:
# common energy and force structures outlayer
for i, dir in enumerate(folders):
    print(dir)
    count=0
    for j in range(len(f_selected_energies[i])):
        if f_structure_unique[i][j] in e_selected_structures[i]:
            count +=1
    print("f_outlayer"  ,len(f_selected_energies[i]))
    print("e_outlayer" ,len(e_selected_structures[i]))
    print("common_outlayer", count)

In [ ]:
#union of energy and forces analysis

tot_union = []
for i in list_common_f:
    tot_union.append(i)
    
for i in list_union_e:
    if i not in list_common_f:
        tot_union.append(i)

#intersection  of energy and forces analysis
intesection = [element for element in list_union_f if element in list_union_e]

print("common force outlayers =" , len(list_common_f), "union energy outlayers =" , len(list_union_e), "tot outlayers =" , len(tot_union))

# np.savetxt( "bad_list_union", union, fmt='%d' )
